In [0]:
%run ./1_raw

In [0]:
print(f"Number of rows in dishes (Bronze): {dishes.shape[0]}")

Number of rows in dishes (Bronze): 231637


# Дивимося страви без імʼя

In [0]:
null_names = dishes['name'].isna().sum()
print(f"No 'name': {null_names}")

No 'name': 1


# Завантаження даних

In [0]:
missing_name_row = dishes[dishes['name'].isna()]
display(missing_name_row)

name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
null,368257,10,779451,2009-04-27,"['15-minutes-or-less', 'time-to-make', 'course', 'preparation', 'low-protein', 'salads', 'easy', 'salad-dressings', 'dietary', 'low-sodium', 'inexpensive', 'low-in-something', '3-steps-or-less']","[1596.2, 249.0, 155.0, 0.0, 2.0, 112.0, 14.0]",6,"['in a bowl , combine ingredients except for olive oil', 'slowly whisk inches', 'olive oil until thickened', 'great with field greens', 'makes about 2 / 3', 'cup dressing']",-------------,"['lemon', 'honey', 'horseradish mustard', 'garlic clove', 'dried parsley', 'dried basil', 'dried thyme', 'garlic salt', 'black pepper', 'olive oil']",10


# Аналіз аутлаєрів по часу 
Перевірка на неможливий час

In [0]:
dishes['minutes'] = pd.to_numeric(dishes['minutes'], errors='coerce')

invalid_minutes = (dishes['minutes'] <= 0).sum()
print(f"time <= 0: {invalid_minutes}")

zero_time_dishes = dishes[dishes['minutes'] == 0]
negative_time_dishes = dishes[dishes['minutes'] < 0]

print(f"Time is 0: {zero_time_dishes.shape[0]}")
print(f"Time less than 0: {negative_time_dishes.shape[0]}")

time <= 0: 1094
Time is 0: 1094
Time less than 0: 0


## Пошук аномально довгих рецептів
Дивимося перцентилі(99) по часу

In [0]:
print("\nPercentiles of 'minutes':")
print(dishes['minutes'].describe(percentiles=[0.5, 0.90, 0.95, 0.99]))

p99 = dishes['minutes'].quantile(0.99)
outlier_dishes = dishes[dishes['minutes'] > p99].copy()

print(f"99 percentile: {p99}")
print(f"number of dishes with minutes > 99 percentile: {outlier_dishes.shape[0]}")


Percentiles of 'minutes':
count    2.316370e+05
mean     9.398546e+03
std      4.461963e+06
min      0.000000e+00
50%      4.000000e+01
90%      1.350000e+02
95%      2.550000e+02
99%      9.039200e+02
max      2.147484e+09
Name: minutes, dtype: float64
99 percentile: 903.9199999999546
number of dishes with minutes > 99 percentile: 2317


## Аналіз відгуків на аутлаєри
Підтягуємо таблицю з відгуками користувачів.
Рахуємо: 
- загальну кількість відгуків
- середній рейтинг

In [0]:
PATH_INTERACTIONS = "/Workspace/Picky_Eater/data/RAW_interactions.csv"
clients_temp = pd.read_csv(PATH_INTERACTIONS, dtype=str)

outlier_ids = outlier_dishes['id'].unique()
outlier_reviews = clients_temp[clients_temp['recipe_id'].isin(outlier_ids)].copy()

print(f"Number of dishes with at least 1 review: {outlier_reviews['recipe_id'].nunique()}")

outlier_reviews['rating'] = pd.to_numeric(outlier_reviews['rating'], errors='coerce')

counts = outlier_reviews['recipe_id'].value_counts().reset_index()
counts.columns = ['recipe_id', 'review_count']

means = outlier_reviews.groupby('recipe_id')['rating'].mean().reset_index()
means.columns = ['recipe_id', 'avg_rating']

stats = pd.merge(counts, means, on='recipe_id')
stats['avg_rating'] = stats['avg_rating'].round(2)

outliers_with_stats = pd.merge(outlier_dishes, stats, left_on='id', right_on='recipe_id', how='left')
outliers_with_stats['review_count'] = outliers_with_stats['review_count'].fillna(0).astype(int)

zero_review_count = (outliers_with_stats['review_count'] == 0).sum()
print(f"Number of dishes with zero reviews: {zero_review_count}")

Number of dishes with at least 1 review: 2317
Number of dishes with zero reviews: 0


# Перевірка на дублікати

dishes full duplicates: 0


Duplicates in 'id': 0
Nulls in 'id': 0


Рахуємо медіани по відгукам та рейтингу

In [0]:
median_reviews = outliers_with_stats['review_count'].median()
median_rating = outlier_reviews['rating'].median()

print(f"Median of reviews: {median_reviews}")
print(f"Median of avarage rating : {median_rating}")

Median of reviews: 2.0
Median of avarage rating : 5.0


## Пошук дивних відгуків
Шукаємо страви, які є фейковими. Відбираємо рецепти з 1 відгуком та перевіряємо, чи це єдиний відгук залишив користувач у всіх рецептах

In [0]:
low_review_outliers = outliers_with_stats[outliers_with_stats['review_count'] <= median_reviews].copy()

low_review_ids = low_review_outliers['id'].unique()
low_reviews_df = outlier_reviews[outlier_reviews['recipe_id'].isin(low_review_ids)].copy()

users_in_low_reviews = low_reviews_df['user_id'].unique()
user_total_reviews = clients_temp[clients_temp['user_id'].isin(users_in_low_reviews)]['user_id'].value_counts().reset_index()
user_total_reviews.columns = ['user_id', 'total_platform_reviews']

low_reviews_detailed = low_reviews_df.merge(user_total_reviews, on='user_id', how='left')

one_time_users_reviews = low_reviews_detailed[low_reviews_detailed['total_platform_reviews'] == 1]
suspicious_recipe_ids = one_time_users_reviews['recipe_id'].unique()

very_suspicious_dishes = low_review_outliers[
    (low_review_outliers['id'].isin(suspicious_recipe_ids)) & 
    (low_review_outliers['review_count'] == 1)
]

print(f"Number of dishes with only 1 review (to outlier dishes): {very_suspicious_dishes.shape[0]}")

Number of dishes with only 1 review (to outlier dishes): 148


Така сама перевірку для страв із 2 відгуками. Шукаємо ситуації, де обидва коментатори - це "одноразові-юзери"

In [0]:
dishes_2_reviews = outliers_with_stats[outliers_with_stats['review_count'] == 2]

reviews_for_2_dishes = low_reviews_detailed[low_reviews_detailed['recipe_id'].isin(dishes_2_reviews['id'])]

max_user_reviews_per_recipe = reviews_for_2_dishes.groupby('recipe_id')['total_platform_reviews'].max().reset_index()
suspicious_ids_2 = max_user_reviews_per_recipe[max_user_reviews_per_recipe['total_platform_reviews'] == 1]['recipe_id']

very_suspicious_2_dishes = dishes_2_reviews[dishes_2_reviews['id'].isin(suspicious_ids_2)]

print(f"Number of dishes with 2 comments from users who have only 1 review (to outlier dishes): {very_suspicious_2_dishes.shape[0]}")

Number of dishes with 2 comments from users who have only 1 review (to outlier dishes): 22


## Список на видалення
Збираємо ID всіх "сміттєвих" страв (з дивними відгуками та з рейтингом менше 3.0) в один список

In [0]:
fake_1_ids_list = very_suspicious_dishes['id'].tolist()
fake_2_ids_list = very_suspicious_2_dishes['id'].tolist()

bad_rating_df = outliers_with_stats[
    (outliers_with_stats['review_count'] > 0) & 
    (outliers_with_stats['avg_rating'] < 3.0)
]
bad_rating_ids_list = bad_rating_df['id'].tolist()

all_bad_outlier_ids = []

for recipe_id in fake_1_ids_list:
    if recipe_id not in all_bad_outlier_ids:
        all_bad_outlier_ids.append(recipe_id)

for recipe_id in fake_2_ids_list:
    if recipe_id not in all_bad_outlier_ids:
        all_bad_outlier_ids.append(recipe_id)

for recipe_id in bad_rating_ids_list:
    if recipe_id not in all_bad_outlier_ids:
        all_bad_outlier_ids.append(recipe_id)

print(f"\nNumber of dishes need to be removed: {len(all_bad_outlier_ids)}")


Number of dishes need to be removed: 376


# Аналіз кроків, інгредієнтів та дати
Перевіряємо на неправильні значення

In [0]:
n_steps_num = pd.to_numeric(dishes['n_steps'], errors='coerce')
n_ingredients_num = pd.to_numeric(dishes['n_ingredients'], errors='coerce')

invalid_steps = ((n_steps_num <= 0) | n_steps_num.isna()).sum()
invalid_ingredients = ((n_ingredients_num <= 0) | n_ingredients_num.isna()).sum()

submitted_date = pd.to_datetime(dishes['submitted'], errors='coerce')
invalid_dates = submitted_date.isna().sum()

print(f"Number of dishes with invalid steps is : {invalid_steps}")
print(f"Number of dishes with invalid ingradients is: {invalid_ingredients}")
print(f"Number of dishes with invalid submishen date: {invalid_dates}")

Number of dishes with invalid steps is : 1
Number of dishes with invalid ingradients is: 0
Number of dishes with invalid submishen date: 0


## Перевірка страви без кроків приготування
Є 1 страва без кроків. Шукаємо її відгуки та перевіряємо, чи готував її хтось, і кількість відгуків цих користувачів

In [0]:
anomaly_recipe_id = dishes[(n_steps_num <= 0) | n_steps_num.isna()]['id'].values[0]
print(f"Reviews by recipe ID: {anomaly_recipe_id}")

anomaly_reviews = clients_temp[clients_temp['recipe_id'] == anomaly_recipe_id]
print(f"Number of reviews: {anomaly_reviews.shape[0]}\n")

anomaly_users = anomaly_reviews['user_id'].unique()
user_review_counts = clients_temp[clients_temp['user_id'].isin(anomaly_users)]['user_id'].value_counts().reset_index()
user_review_counts.columns = ['user_id', 'total_reviews']

display(user_review_counts)
display(anomaly_reviews[['user_id', 'date', 'rating', 'review']])

anomaly_row = dishes[(n_steps_num <= 0) | (n_steps_num.isna())]
display(anomaly_row)

Reviews by recipe ID: 176767
Number of reviews: 2



user_id,total_reviews
233583,522
38218,220


user_id,date,rating,review
38218,2006-08-26,5,"This is a little bit different from my usual recipie (which I've lost). but it makes a lovely, fragrant loaf. Sublime with a smear of cream cheese. I've only tried the carrot so far, but that was such a success I can't wait to try the others. I made as directed with rasins and walnuts."
233583,2010-04-23,4,Made for PAC 2010 What can you say about a bread that is so versatile and readily available with ingredients on hand but YUMMO! Made this yesterday for a snack and a healthier breakfast on the run. I used bananas and shredded carrots and mmmm mmmm it came out very tastey. So 1/2 disappeared before pictures could be taken but did get a shot this morning of the 1 loaf and a few muffins left over. It came out so good I made a 2nd batch to take to the cottage for the weekend. Quite a moist bread/muffin. I believe I will make the zucchini variety next week as I still have some frozen from last summer harvest so out fo the freezer it comes to defrost and drain. Thank you.


name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
all season bread,176767,90,331268,2006-07-10,"['time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'for-large-groups', 'breads', 'fruit', 'vegetables', 'easy', 'fall', 'spring', 'winter', 'muffins', 'coffee-cakes', 'seasonal', 'comfort-food', 'inexpensive', 'quick-breads', 'tropical-fruit', 'bananas', 'carrots', 'taste-mood', 'number-of-servings', '3-steps-or-less', '4-hours-or-less']","[198.8, 11.0, 70.0, 18.0, 5.0, 5.0, 10.0]",0,[],"just change the fruit/vegetable in this recipe and make the (tender, moist, heavy dark) bread your heart desires! try zucchini in the summer, pumpkin in the fall, carrot-raisin in the winter, & banana-walnut in the spring. use your imagination.","['flour', 'baking soda', 'salt', 'baking powder', 'cinnamon', 'eggs', 'white sugar', 'vegetable oil', 'real vanilla', 'raw carrots', 'raisins', 'walnuts']",12


# Final Pipeline
Фінальний етап очищення. 
1. **Name та Minutes:** Видаляємо пусті ID та страви, які ми знайшли раніше (одноразові-коментатори + низький рейтинг).
2. **Steps та Submitted Date:** Якщо кроки або дата відсутні, ми дивимося на відгуки. Якщо страва має відгуки з середнім рейтингом 3.0 і вище — ми її зберігаємо (ставимо заглушку `['0000']` для кроків та `1900-01-01` для дати). Якщо відгуків немає або рейтинг низький — видаляємо.
3. **Ingredients:** 100% видаляємо страви без інгредієнтів, оскільки з ними неможливо працювати в рекомендаційній системі.

In [0]:
dishes_silver = dishes.copy()

dishes_silver['name'] = dishes_silver['name'].fillna("No Name")
dishes_silver = dishes_silver[dishes_silver['id'].notna()]

dishes_silver['minutes'] = pd.to_numeric(dishes_silver['minutes'], errors='coerce')
dishes_silver = dishes_silver[
    (dishes_silver['minutes'] > 0) & 
    (~dishes_silver['id'].isin(all_bad_outlier_ids))
]

dishes_silver['n_steps'] = pd.to_numeric(dishes_silver['n_steps'], errors='coerce')

empty_steps_mask = dishes_silver['steps'].isna() | (dishes_silver['steps'] == '[]') | (dishes_silver['steps'] == '')
empty_steps_ids = dishes_silver[empty_steps_mask]['id'].tolist()

reviews_for_empty_steps = clients_temp[clients_temp['recipe_id'].isin(empty_steps_ids)].copy()
reviews_for_empty_steps['rating'] = pd.to_numeric(reviews_for_empty_steps['rating'], errors='coerce')

ids_to_drop_steps = []

for recipe_id in empty_steps_ids:
    recipe_reviews = reviews_for_empty_steps[reviews_for_empty_steps['recipe_id'] == recipe_id]
    count = recipe_reviews.shape[0]
    
    if count > 0:
        avg_rating = recipe_reviews['rating'].mean()
    else:
        avg_rating = 0.0
        
    if count > 0 and avg_rating >= 3.0:
        dishes_silver.loc[dishes_silver['id'] == recipe_id, 'steps'] = "['0000']"
        dishes_silver.loc[dishes_silver['id'] == recipe_id, 'n_steps'] = 0
    else:
        ids_to_drop_steps.append(recipe_id)

dishes_silver = dishes_silver[~dishes_silver['id'].isin(ids_to_drop_steps)]

dishes_silver['n_ingredients'] = pd.to_numeric(dishes_silver['n_ingredients'], errors='coerce')
empty_ingredients = dishes_silver['ingredients'].isna() | (dishes_silver['ingredients'] == '[]') | (dishes_silver['ingredients'] == '')
invalid_n_ingredients = dishes_silver['n_ingredients'].isna() | (dishes_silver['n_ingredients'] <= 0)

dishes_silver = dishes_silver[~(invalid_n_ingredients & empty_ingredients)]

dishes_silver['submitted'] = pd.to_datetime(dishes_silver['submitted'], errors='coerce')

empty_dates_mask = dishes_silver['submitted'].isna()
empty_dates_ids = dishes_silver[empty_dates_mask]['id'].tolist()

reviews_for_empty_dates = clients_temp[clients_temp['recipe_id'].isin(empty_dates_ids)].copy()
reviews_for_empty_dates['rating'] = pd.to_numeric(reviews_for_empty_dates['rating'], errors='coerce')

ids_to_drop_dates = []

for recipe_id in empty_dates_ids:
    recipe_reviews = reviews_for_empty_dates[reviews_for_empty_dates['recipe_id'] == recipe_id]
    count = recipe_reviews.shape[0]
    
    if count > 0:
        avg_rating = recipe_reviews['rating'].mean()
    else:
        avg_rating = 0.0
        
    if count > 0 and avg_rating >= 3.0:
        dishes_silver.loc[dishes_silver['id'] == recipe_id, 'submitted'] = pd.to_datetime('1900-01-01')
    else:
        ids_to_drop_dates.append(recipe_id)

dishes_silver = dishes_silver[~dishes_silver['id'].isin(ids_to_drop_dates)]

print(f"Final number of rows in dishes_silver: {dishes_silver.shape[0]}")

Final number of rows in dishes_silver: 230167
